# TEST5000 — chuẩn bị ROI và prompt, chưa đánh giá caption
**Accelerator None**, Internet ON. Không cần Secret/checkpoint. CPU pooling từ visual-cache, không chạy lại CLIP ảnh/UPT; chỉ encode CPU các text chưa có trong175.
Gắn5Dataset: COCO2014(cảJPEG+JSON), visual-cache, upt-prompts-full-all-actions-v2, objectdetectionecache, csa-prompt-full-karpathy-v175. Chỉnh path ở cell đầu nếu mount khác.
Giữ nguyên KarpathyTEST5000, ROI/profile/vocab và ngưỡng cũ. Không dùng caption tham chiếu để tạo prompt; prompt thiếu ảnh sẽ dừng. Metadata tham chiếu chỉ giữ riêng cho evaluator sau này.
Giữ cả output và gửi `csa_prompt_test_preparation_v1_review.zip`. Bước này cần trước vì cache175 không chứaTEST. Sau khi xác minh receipt/coverage mới chuẩn bị lượt caption TEST C180/P60 và normal/zero/shuffle theo protocol đã khóa.
Tác giả chỉ kiểm metadata/static/fixture; chưa chạy notebook hoặc model. Không tiêu GPU nếu chọn Accelerator None.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile
from pathlib import Path
BASE=Path('/kaggle/input/datasets')
COCO=BASE/'vuthetam/mscoco-2014'
VISUAL=BASE/'ducanh2403/visual-cache'
PROMPTS=BASE/'ducanh2403/upt-prompts-full-all-actions-v2/prompt_cache.json'
DETECTIONS=BASE/'ducanh2403/objectdetectionecache/objectdetectioncache.json'
PROMPT175=BASE/'ducanh2403/csa-prompt-full-karpathy-v175'
OUTPUT=Path('/kaggle/working/csa_prompt_test_preparation_v1')
for p in (COCO/'dataset_coco.json',VISUAL,PROMPTS,DETECTIONS,PROMPT175):
    if not p.exists():raise FileNotFoundError('Attach input/correct path: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve prior output; use fresh runtime')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-prompt-test-prepare-',dir='/tmp'));LOG=OUTPUT/'notebook.log'
REVISION='f3a8a658c5cce50e879c4d905bf46211420e0b5d'
MAPPING={'experiments/csa_paper/full_roi_review_text/input_receipt.json': 'full_roi_review_text/input_receipt.json', 'experiments/csa_paper/SAVE_XE139_PACKAGE.json': 'SAVE_XE139_PACKAGE.json', 'experiments/csa_paper/BEST9_SAVED_TEXT_PACKAGE.json': 'BEST9_SAVED_TEXT_PACKAGE.json', 'experiments/csa_paper/best9_saved_text.json.gz': 'best9_saved_text.json.gz', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'SAVE_FULL_ROI_V134_PACKAGE.json', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': 'RERANK_TEST_INPUTS.json', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': 'FULL_PROMPT_FRESH_V175_RESULT.json', 'experiments/csa_paper/PROMPT_TEST_PROTOCOL.md': 'PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/make_prompt_test_preparation.py': 'make_prompt_test_preparation.py', 'experiments/csa_paper/make_smoke_notebook.py': 'make_smoke_notebook.py', 'experiments/csa_paper/make_prompt_scst_notebooks.py': 'make_prompt_scst_notebooks.py', 'experiments/csa_paper/test_prompt_test_preparation.py': 'test_prompt_test_preparation.py', 'experiments/csa_paper/prepare_prompt_test.py': 'prepare_prompt_test.py', 'experiments/csa_paper/prompt_text_cache.py': 'prompt_text_cache.py', 'experiments/csa_paper/prompt_adapter_data.py': 'prompt_adapter_data.py', 'captioning/upt_prompts.py': 'captioning/upt_prompts.py', 'captioning/upt_action_text.py': 'captioning/upt_action_text.py', 'captioning/selected_relation_prompts.py': 'captioning/selected_relation_prompts.py', 'captioning/supervised_relations.py': 'captioning/supervised_relations.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/prepare_test_regions.py': 'prepare_test_regions.py', 'experiments/csa_paper/run_best9_decode.py': 'run_best9_decode.py', 'experiments/csa_paper/model.py': 'model.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/prepare_real_assets.py': 'prepare_real_assets.py', 'experiments/csa_paper/audit_assets.py': 'audit_assets.py', 'experiments/csa_paper/prompt_scratch_data.py': 'prompt_scratch_data.py', 'experiments/csa_paper/full_prompt_cache_data.py': 'full_prompt_cache_data.py', 'experiments/csa_paper/sharing_data.py': 'sharing_data.py', 'experiments/csa_paper/full_cache_data.py': 'full_cache_data.py', 'experiments/csa_paper/package_full_roi_dataset.py': 'package_full_roi_dataset.py', 'captioning/__init__.py': 'captioning/__init__.py'}
HASHES={'experiments/csa_paper/full_roi_review_text/input_receipt.json': '18e9481863418c7f2f4c538fa98f3a05377e5b81422cb249d33f81ddefd78f7f', 'experiments/csa_paper/SAVE_XE139_PACKAGE.json': '452ee35118a3d0285bc7f522bb8c8fc7a0e3e0dac70cac9c2333e7583cd4e423', 'experiments/csa_paper/BEST9_SAVED_TEXT_PACKAGE.json': '898d67c3eead66c89f5d2c091b24e5e3eb44d884e86550b2b26f45d6708032e5', 'experiments/csa_paper/best9_saved_text.json.gz': '42594e90ec2f47646e7e68347a17471213038bc182814ca9d29ea374e8010861', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a', 'experiments/csa_paper/PROMPT_TEST_PROTOCOL.md': '002277c7dab3880148d1d0e8c600ca117be78734f85b55c6dd8bf13d50485c70', 'experiments/csa_paper/make_prompt_test_preparation.py': 'f1e311b0fdce42c7b0b2afedf16665a91c313f0fbeefd1a34c0009665a4c7e25', 'experiments/csa_paper/make_smoke_notebook.py': '9de13e286efb3ddd29a2eed331219a9e60dcb384fc2f28e062f64dd8dd1c6054', 'experiments/csa_paper/make_prompt_scst_notebooks.py': 'cc5f491c5da80c1615736198a65d6f5b5259455867ee4dc4f6a8e55f6d5cde6c', 'experiments/csa_paper/test_prompt_test_preparation.py': 'fae132ef961b34050771807fe6682b2d09f7a4366e108842313bac66beaf0e71', 'experiments/csa_paper/prepare_prompt_test.py': 'ba571bcfc19fffe26ac5145c91c76143ccb8cfff01a20c1c8210da9fcfaf4a20', 'experiments/csa_paper/prompt_text_cache.py': '0bd52c07b44d201a631e6383af111c21d7b3ef2e2ed676f5bc13fc53306e0f23', 'experiments/csa_paper/prompt_adapter_data.py': 'd8b8b61a518d9d4e2d413386d08e60963586bf3b17a16e5c96ee3f54eeaf36c0', 'captioning/upt_prompts.py': '25bb24346d2b2c649ee7091c72d5def060ddf2c290524a35ef3d301d274fb65c', 'captioning/upt_action_text.py': 'ca0d8f1dc75be3ccfd589fcddc962e87a43222751a4d1beb68098cfa213d526c', 'captioning/selected_relation_prompts.py': 'de13dfbd80e4d4b8c08cd3e40407d5134cb1a5108c58f3eebfd4b74e237179d2', 'captioning/supervised_relations.py': '3e29fd60bbf166ae078581fb9ad34da6ce4462ee287f1c4eb9bbda2abd91f984', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/prepare_test_regions.py': '3c61ed502f3091293c537a32ec2c0f82c72f4bf97cc86cfabe4b035c365692b7', 'experiments/csa_paper/run_best9_decode.py': 'fdc6b027ac13d510be72aa25a963eb89e5467e09f179bfaf0756469aef77a648', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/prepare_real_assets.py': '44a22d92acd239bdc7a1edbe30984daf0853ac8bc196969dd48b19b78df1cc7b', 'experiments/csa_paper/audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959', 'experiments/csa_paper/prompt_scratch_data.py': '025496a549e12a49104992e1c48f09968ebf4dcb520d649c818df6b39c0a6959', 'experiments/csa_paper/full_prompt_cache_data.py': 'efb21689e1983e8ab22f9d63068cf262f58344498d412056580c8af54c481f16', 'experiments/csa_paper/sharing_data.py': '68a065f5c055a55b41eae0ea554fbb7f8fd1124e8f5422679fcb1d4878f14829', 'experiments/csa_paper/full_cache_data.py': '58cf4d08310fe03e01441af2ddeb12fd3404f59e82259194a32fe98c8b002bc1', 'experiments/csa_paper/package_full_roi_dataset.py': 'be3e7d0f96add450a5295add6466e19977da90f486e5e726e3ea40a505cceecb', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633'}

for source,dest in MAPPING.items():
    raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+source,timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=HASHES[source]:raise ValueError('Changed pinned file:'+source)
    p=SOURCE/dest;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2),encoding='utf-8')
DEPS=SOURCE/'deps';os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
os.environ['OMP_NUM_THREADS']='2'
CONFIG=SOURCE/'config.json'
CONFIG.write_text(json.dumps(dict(output=str(OUTPUT),dataset_json=str(COCO/'dataset_coco.json'),detections=str(DETECTIONS),
    visual_cache=str(VISUAL),image_root=str(COCO/'images' if (COCO/'images').is_dir() else COCO),
    prompts=str(PROMPTS),prompt_root=str(PROMPT175),hf_cache=str(SOURCE/'hf-cache'))),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Smoke active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'test_prompt_test_preparation.py'],timeout=120)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'transformers==4.57.1','tokenizers==0.22.1','huggingface-hub==0.36.2','safetensors==0.6.2'],timeout=300)
    run([sys.executable,'-u',SOURCE/'prepare_prompt_test.py','--config',CONFIG],timeout=7200)
    complete=json.loads((OUTPUT/'summary.json').read_text(encoding='utf-8'))['status']=='PROMPT_TEST_PREPARATION_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,caption_generation=False,caption_training=False,caption_scoring=False)),encoding='utf-8')
    REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.iterdir():
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt') and p.name!='test_metadata.json':z.write(p,p.name)
    print('SEND REVIEW:',REVIEW,'\nKEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('PREPARATION COMPLETE; send review before caption evaluation.' if complete else 'INCOMPLETE; send log/review, no substitute prompt sources.',flush=True)
